# 03 — Held-out simulation evaluation

This notebook evaluates the frozen MLP trained with dynamic Rician noise over SNR 30–70.

The held-out synthetic test partition is used to assess:

1. performance across fixed SNR conditions;
2. model-specific recall across SNR;
3. performance under variable SNR 30–70;
4. relative classifier confidence using maximum Softmax score and top-two probability margin.

The held-out test partition contains 180,000 signals, with 20,000 from each of the nine model classes.

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results" / "simulation"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

X_CLEAN_FILE = DATA_DIR / "X_clean.pt"
Y_FILE = DATA_DIR / "y_labels.pt"
SPLIT_FILE = DATA_DIR / "master_split_80train_10val_10test_seed42.npz"
TRUE_PARAMS_FILE = DATA_DIR / "true_parameters.pt"

STATE_FILE = CHECKPOINT_DIR / "MLP_FRESH_RANDOM_SNR30_70_state_dict.pt"
STATS_FILE = CHECKPOINT_DIR / "MLP_FRESH_RANDOM_SNR30_70_feature_standardisation.pt"

for path in [X_CLEAN_FILE, Y_FILE, SPLIT_FILE, STATE_FILE, STATS_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
metadata = torch.load(METADATA_FILE, map_location="cpu", weights_only=False)
CLASS_NAMES = metadata["model_order"]
FIXED_SNR_VALUES = [100, 75, 50, 25, 10]
NOISE_SEEDS = [1001, 1002, 1003]

VARIABLE_SNR_LOW = 30.0
VARIABLE_SNR_HIGH = 70.0

PREDICTION_BATCH_SIZE = 8192

FIXED_CONDITION_ORDER = ["Clean", "SNR100", "SNR75", "SNR50", "SNR25", "SNR10"]

NameError: name 'torch' is not defined

In [ ]:
class FinalBrainModelMLP(nn.Module):
    def __init__(
        self,
        input_dim=131,
        num_classes=9,
        hidden_dims=(512, 512, 256, 128),
        dropout=0.05,
    ):
        super().__init__()

        layers = []
        previous = input_dim

        for hidden in hidden_dims:
            layers.extend([
                nn.Linear(previous, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            previous = hidden

        layers.append(nn.Linear(previous, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# Load the frozen MLP.
state_dict = torch.load(STATE_FILE, map_location="cpu", weights_only=True)

model = FinalBrainModelMLP(num_classes=len(CLASS_NAMES)).to(device)
model.load_state_dict(state_dict)
model.eval()

# Load the feature-standardisation statistics.
stats = torch.load(STATS_FILE, map_location="cpu", weights_only=True)

feature_mean = stats["feature_mean"].float().reshape(1, -1)
feature_std = stats["feature_std"].float().reshape(1, -1)

In [ ]:
y_all = torch.load(Y_FILE, map_location="cpu", weights_only=True).long()

with np.load(SPLIT_FILE) as split:
    test_idx = split["test_idx"].astype(np.int64)

test_idx_t = torch.from_numpy(test_idx)

y_test = y_all.index_select(0, test_idx_t).numpy()

X_clean_full = torch.load(X_CLEAN_FILE, map_location="cpu", weights_only=True).float()

X_test_clean = X_clean_full.index_select(0, test_idx_t).contiguous()

del X_clean_full, y_all
gc.collect()

assert tuple(X_test_clean.shape) == (180_000, 131)

In [ ]:
def add_fixed_rician_noise(x_clean, snr, seed, chunk_size=20_000):
    sigma = 1.0 / snr

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    x_noisy = torch.empty_like(x_clean)

    for start in range(0, len(x_clean), chunk_size):
        end = min(start + chunk_size, len(x_clean))
        clean = x_clean[start:end]

        real_noise = torch.randn(
            clean.shape,
            dtype=clean.dtype,
            generator=generator,
        ) * sigma

        imag_noise = torch.randn(
            clean.shape,
            dtype=clean.dtype,
            generator=generator,
        ) * sigma

        x_noisy[start:end] = torch.sqrt(
            (clean + real_noise) ** 2 + imag_noise ** 2
        )

    return x_noisy


def add_variable_snr_rician_noise(
    x_clean,
    snr_low,
    snr_high,
    seed,
    chunk_size=20_000,
):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    x_noisy = torch.empty_like(x_clean)

    for start in range(0, len(x_clean), chunk_size):
        end = min(start + chunk_size, len(x_clean))
        clean = x_clean[start:end]
        n = end - start

        snr_values = (
            torch.rand(
                (n, 1),
                dtype=clean.dtype,
                generator=generator,
            )
            * (snr_high - snr_low)
            + snr_low
        )

        sigma = 1.0 / snr_values

        real_noise = torch.randn(
            clean.shape,
            dtype=clean.dtype,
            generator=generator,
        ) * sigma

        imag_noise = torch.randn(
            clean.shape,
            dtype=clean.dtype,
            generator=generator,
        ) * sigma

        x_noisy[start:end] = torch.sqrt(
            (clean + real_noise) ** 2 + imag_noise ** 2
        )

    return x_noisy

In [ ]:
@torch.no_grad()
def predict_probabilities(X):
    loader = DataLoader(
        X,
        batch_size=PREDICTION_BATCH_SIZE,
        shuffle=False,
    )

    mean_d = feature_mean.to(device)
    std_d = feature_std.to(device)

    probability_batches = []

    for xb in loader:
        xb = xb.to(device)
        xb = (xb - mean_d) / std_d

        logits = model(xb)
        probabilities = torch.softmax(logits, dim=1)
        probability_batches.append(probabilities.cpu())

    return torch.cat(probability_batches).numpy()

## Held-out SNR robustness

Clean evaluation is deterministic and is performed once. Each noisy SNR condition is evaluated across the three predefined Rician noise realisations with seeds `1001`, `1002`, and `1003`.

In [ ]:
metric_rows = []
class_rows = []


def evaluate_matrix(X_eval, condition, noise_seed=None):
    probabilities = predict_probabilities(X_eval)
    y_pred = probabilities.argmax(axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0,
    )

    metric_rows.append({
        "condition": condition,
        "noise_seed": noise_seed,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    })

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
    )

    recall = np.diag(cm) / cm.sum(axis=1)

    for class_name, class_recall in zip(CLASS_NAMES, recall):
        class_rows.append({
            "condition": condition,
            "noise_seed": noise_seed,
            "class": class_name,
            "recall": class_recall,
        })


# Clean held-out evaluation.
evaluate_matrix(
    X_test_clean,
    condition="Clean",
)

# Fixed-SNR held-out evaluation.
for snr in FIXED_SNR_VALUES:
    for seed in NOISE_SEEDS:
        X_eval = add_fixed_rician_noise(
            X_test_clean,
            snr=snr,
            seed=seed,
        )

        evaluate_matrix(
            X_eval,
            condition=f"SNR{snr}",
            noise_seed=seed,
        )

        del X_eval
        gc.collect()


run_metrics = pd.DataFrame(metric_rows)
class_metrics = pd.DataFrame(class_rows)

run_metrics.to_csv(
    TABLE_DIR / "mlp_snr_run_metrics.csv",
    index=False,
)

class_metrics.to_csv(
    TABLE_DIR / "mlp_snr_per_class_recall.csv",
    index=False,
)

display(run_metrics)

In [ ]:
recall_summary = (
    class_metrics
    .groupby(["condition", "class"], as_index=False)
    ["recall"]
    .mean()
)

recall_summary["condition"] = pd.Categorical(
    recall_summary["condition"],
    categories=FIXED_CONDITION_ORDER,
    ordered=True,
)

recall_summary = (
    recall_summary
    .sort_values(["class", "condition"])
    .reset_index(drop=True)
)

recall_summary.to_csv(
    TABLE_DIR / "mlp_recall_by_snr.csv",
    index=False,
)


recall_matrix = (
    recall_summary
    .pivot(index="class", columns="condition", values="recall")
    .reindex(index=CLASS_NAMES, columns=FIXED_CONDITION_ORDER)
)

fig, ax = plt.subplots(figsize=(8, 6))

image = ax.imshow(recall_matrix.to_numpy() * 100, aspect="auto")

ax.set_xticks(np.arange(len(FIXED_CONDITION_ORDER)))
ax.set_xticklabels(FIXED_CONDITION_ORDER)

ax.set_yticks(np.arange(len(CLASS_NAMES)))
ax.set_yticklabels(CLASS_NAMES)

ax.set_xlabel("Evaluation condition")
ax.set_ylabel("Generating model")
ax.set_title("Held-out model-specific recall (%)")

for i in range(len(CLASS_NAMES)):
    for j in range(len(FIXED_CONDITION_ORDER)):
        value = recall_matrix.iloc[i, j] * 100
        ax.text(j, i, f"{value:.1f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="Recall (%)")
fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "mlp_recall_by_snr.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "mlp_recall_by_snr.pdf",
    bbox_inches="tight",
)

plt.show()

## Variable-SNR evaluation and classifier confidence

The MLP was trained with dynamic Rician noise using SNR sampled from 30–70. The next analysis evaluates three held-out variable-SNR realisations generated from the same range.

Maximum Softmax score and the top-two Softmax margin are used as measures of relative classifier confidence and ambiguity, not biological probabilities.

Model-specific confidence is summarised by the median maximum Softmax score and median top-two margin for each generating–predicted model combination.

In [ ]:
confidence_rows = []
variable_metric_rows = []

for seed in NOISE_SEEDS:
    X_eval = add_variable_snr_rician_noise(
        X_test_clean,
        snr_low=VARIABLE_SNR_LOW,
        snr_high=VARIABLE_SNR_HIGH,
        seed=seed,
    )

    probabilities = predict_probabilities(X_eval)
    y_pred = probabilities.argmax(axis=1)

    sorted_probabilities = np.sort(probabilities, axis=1)
    max_softmax = sorted_probabilities[:, -1]
    top2_margin = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]

    variable_metric_rows.append({
        "noise_seed": seed,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        ),
    })

    confidence_rows.append(
        pd.DataFrame({
            "noise_seed": seed,
            "true_class": y_test,
            "predicted_class": y_pred,
            "max_softmax": max_softmax,
            "top2_margin": top2_margin,
        })
    )

    del X_eval, probabilities, sorted_probabilities
    gc.collect()


variable_metrics = pd.DataFrame(variable_metric_rows)
confidence_df = pd.concat(confidence_rows, ignore_index=True)

variable_metrics.to_csv(
    TABLE_DIR / "mlp_variable_snr30_70_metrics.csv",
    index=False,
)

display(variable_metrics)

In [ ]:
SHORT_NAMES = ["Ball", "BS", "ZS", "TS", "BSS", "ZBS", "ZAS", "TBS", "TAS"]

N_CLASSES = len(CLASS_NAMES)

# Mask very rare true-predicted combinations in the heatmap for better visualization.

MIN_COUNT = 100

count_matrix = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
max_softmax_matrix = np.full((N_CLASSES, N_CLASSES), np.nan)
top2_margin_matrix = np.full((N_CLASSES, N_CLASSES), np.nan)

for true_class in range(N_CLASSES):
    for predicted_class in range(N_CLASSES):
        mask = (
            (confidence_df["true_class"] == true_class)
            & (confidence_df["predicted_class"] == predicted_class)
        )

        n = int(mask.sum())
        count_matrix[true_class, predicted_class] = n

        if n >= MIN_COUNT:
            max_softmax_matrix[true_class, predicted_class] = (
                confidence_df.loc[mask, "max_softmax"].median()
            )

            top2_margin_matrix[true_class, predicted_class] = (
                confidence_df.loc[mask, "top2_margin"].median()
            )


pd.DataFrame(
    count_matrix,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(
    TABLE_DIR / "mlp_confidence_pair_counts.csv"
)

pd.DataFrame(
    max_softmax_matrix,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(
    TABLE_DIR / "mlp_median_max_softmax_by_model_pair.csv"
)

pd.DataFrame(
    top2_margin_matrix,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(
    TABLE_DIR / "mlp_median_top2_margin_by_model_pair.csv"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.3))

cmap = plt.cm.RdYlGn.copy()
cmap.set_bad("whitesmoke")


def draw_heatmap(ax, matrix, title):
    image = ax.imshow(
        matrix,
        cmap=cmap,
        vmin=0,
        vmax=1,
        aspect="equal",
    )

    ax.set_xticks(np.arange(N_CLASSES))
    ax.set_xticklabels(SHORT_NAMES, rotation=45, ha="right")

    ax.set_yticks(np.arange(N_CLASSES))
    ax.set_yticklabels(SHORT_NAMES)

    ax.set_xlabel("Predicted model")
    ax.set_title(title)

    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            value = matrix[i, j]

            if np.isnan(value):
                continue

            rgba = image.cmap(image.norm(value))
            luminance = (
                0.2126 * rgba[0]
                + 0.7152 * rgba[1]
                + 0.0722 * rgba[2]
            )

            ax.text(
                j,
                i,
                f"{100 * value:.0f}",
                ha="center",
                va="center",
                color="white" if luminance < 0.48 else "black",
            )

    return image


image = draw_heatmap(
    axes[0],
    max_softmax_matrix,
    "(A) Median maximum Softmax score",
)

axes[0].set_ylabel("Generating model")

draw_heatmap(
    axes[1],
    top2_margin_matrix,
    "(B) Median top-two Softmax margin",
)

fig.colorbar(
    image,
    ax=axes,
    label="Median score",
    fraction=0.03,
    pad=0.03,
)

fig.suptitle(
    "Model-specific confidence and ambiguity on held-out variable-SNR data"
)

fig.savefig(
    FIGURE_DIR / "mlp_model_specific_confidence.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR / "mlp_model_specific_confidence.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
## Parameter-dependent ambiguity

Structured classification errors were examined in parameter regimes where competing model structures become difficult to distinguish.

First, ZeppelinSphere-to-BallSphere confusion is examined as the Zeppelin approaches the isotropic Ball limit. Clean predictions are compared with variable-SNR30–70 predictions pooled across the three held-out noise realisations.

Second, model selection is examined across the three-compartment fraction space for the minimum-fraction 0.01 simulation design. Dominant predictions are pooled across the same three variable-SNR30–70 realisations.

In [ ]:
true_parameters = torch.load(
    TRUE_PARAMS_FILE,
    map_location="cpu",
    weights_only=False,
)

variable_pred_by_seed = {
    seed: confidence_df.loc[
        confidence_df["noise_seed"] == seed,
        "predicted_class",
    ].to_numpy()
    for seed in NOISE_SEEDS
}

# Clean predictions for the Zeppelin–Ball comparison.
clean_probabilities = predict_probabilities(X_test_clean)
clean_pred = clean_probabilities.argmax(axis=1)

del clean_probabilities
gc.collect()

In [ ]:
ZS_IDX = CLASS_NAMES.index("ZeppelinSphere")
BS_IDX = CLASS_NAMES.index("BallSphere")

zs_mask = y_test == ZS_IDX
zs_global_indices = test_idx[zs_mask]

start, end = true_parameters["row_ranges"]["ZeppelinSphere"]
local_indices = zs_global_indices - int(start)

zeppelin = true_parameters["ZeppelinSphere"]["zeppelin"]

Dpar = torch.as_tensor(zeppelin["Dpar"]).reshape(-1)[local_indices].numpy()
Dperp = torch.as_tensor(zeppelin["Dperp"]).reshape(-1)[local_indices].numpy()

zeppelin_anisotropy = (Dpar - Dperp) / Dpar


def binned_directional_error(
    parameter_values,
    prediction_arrays,
    target_class,
    bin_edges,
):
    rows = []

    for i in range(len(bin_edges) - 1):
        left = bin_edges[i]
        right = bin_edges[i + 1]

        if i < len(bin_edges) - 2:
            bin_mask = (
                (parameter_values >= left)
                & (parameter_values < right)
            )
        else:
            bin_mask = (
                (parameter_values >= left)
                & (parameter_values <= right)
            )

        n_unique = int(bin_mask.sum())

        confused = sum(
            int(np.sum(predictions[bin_mask] == target_class))
            for predictions in prediction_arrays
        )

        n_predictions = n_unique * len(prediction_arrays)

        rows.append({
            "left": left,
            "right": right,
            "mid": (left + right) / 2,
            "n_unique": n_unique,
            "n_predictions": n_predictions,
            "confused": confused,
            "rate": confused / n_predictions if n_predictions else np.nan,
        })

    return pd.DataFrame(rows)


anisotropy_bins = np.linspace(0, 1, 11)

zs_bs_clean = binned_directional_error(
    zeppelin_anisotropy,
    [clean_pred[zs_mask]],
    BS_IDX,
    anisotropy_bins,
)

zs_bs_variable = binned_directional_error(
    zeppelin_anisotropy,
    [
        variable_pred_by_seed[seed][zs_mask]
        for seed in NOISE_SEEDS
    ],
    BS_IDX,
    anisotropy_bins,
)

zs_bs_clean.to_csv(
    TABLE_DIR / "zs_to_bs_anisotropy_clean.csv",
    index=False,
)

zs_bs_variable.to_csv(
    TABLE_DIR / "zs_to_bs_anisotropy_variable_snr30_70.csv",
    index=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))

ax.plot(
    zs_bs_variable["mid"],
    100 * zs_bs_variable["rate"],
    marker="o",
    linewidth=2,
    label="Variable SNR 30–70",
)

ax.plot(
    zs_bs_clean["mid"],
    100 * zs_bs_clean["rate"],
    marker="o",
    linestyle="--",
    linewidth=1.7,
    markerfacecolor="white",
    label="Clean",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 100)

ax.set_xlabel("Zeppelin anisotropy (D∥ − D⊥) / D∥")
ax.set_ylabel("ZeppelinSphere → BallSphere misclassification rate (%)")
ax.set_title("Dependence of Zeppelin–Ball ambiguity on Zeppelin anisotropy")

ax.grid(alpha=0.25)
ax.legend(frameon=False)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "zeppelin_ball_ambiguity.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR / "zeppelin_ball_ambiguity.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon, Patch


THREE_COMPARTMENT_MODELS = [
    "BallSphereStick",
    "ZeppelinBallSphere",
    "ZeppelinAstrosticksSphere",
    "TensorBallSphere",
    "TensorAstrosticksSphere",
]

SHORT_NAME_MAP = {
    "Ball": "Ball",
    "BallSphere": "BS",
    "ZeppelinSphere": "ZS",
    "TensorSphere": "TS",
    "BallSphereStick": "BSS",
    "ZeppelinBallSphere": "ZBS",
    "ZeppelinAstrosticksSphere": "ZAS",
    "TensorBallSphere": "TBS",
    "TensorAstrosticksSphere": "TAS",
}

TERNARY_N_BINS = 20

# Plotting-only threshold for sparsely populated fraction-space bins.
PLOT_MIN_UNIQUE_SAMPLES = 5

colour_map = plt.get_cmap("tab10")
CLASS_COLOURS = [
    np.asarray(colour_map(i)[:3])
    for i in range(len(CLASS_NAMES))
]


def get_three_compartment_fractions(true_parameters, model_name):
    fraction_object = true_parameters[model_name]["fractions"]

    if "fractions" in fraction_object:
        fractions = torch.as_tensor(
            fraction_object["fractions"]
        ).cpu().numpy()
    else:
        fractions = np.column_stack([
            torch.as_tensor(fraction_object["f_ees"]).reshape(-1).numpy(),
            torch.as_tensor(fraction_object["f_vasc"]).reshape(-1).numpy(),
            torch.as_tensor(fraction_object["f_ic"]).reshape(-1).numpy(),
        ])

    return np.asarray(fractions, dtype=float)


def ternary_xy(f_ees, f_vasc, f_ic):
    h = np.sqrt(3) / 2

    x = f_vasc + 0.5 * f_ic
    y = h * f_ic

    return np.asarray(x), np.asarray(y)


def clip_polygon_to_simplex(polygon, tolerance=1e-12):
    def inside(point):
        return point[0] + point[1] <= 1 + tolerance

    def intersection(a, b):
        sum_a = a[0] + a[1]
        sum_b = b[0] + b[1]
        t = (1 - sum_a) / (sum_b - sum_a)
        return a + t * (b - a)

    output = []
    previous = polygon[-1]
    previous_inside = inside(previous)

    for current in polygon:
        current_inside = inside(current)

        if current_inside:
            if not previous_inside:
                output.append(intersection(previous, current))
            output.append(current)

        elif previous_inside:
            output.append(intersection(previous, current))

        previous = current
        previous_inside = current_inside

    return np.asarray(output, dtype=float)


def draw_ternary_frame(ax):
    h = np.sqrt(3) / 2

    border = np.array([
        [0, 0],
        [1, 0],
        [0.5, h],
        [0, 0],
    ])

    ax.plot(border[:, 0], border[:, 1], linewidth=1.5)

    ax.text(0.50, -0.10, "fEES", ha="center", va="top")
    ax.text(0.10, 0.47, "fIC", rotation=60, ha="center")
    ax.text(0.90, 0.47, "fVASC", rotation=-60, ha="center")

    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.12, h + 0.06)
    ax.set_aspect("equal")
    ax.axis("off")

In [ ]:
def plot_fraction_space(ax, true_model):
    true_class = CLASS_NAMES.index(true_model)
    model_mask = y_test == true_class

    model_test_indices = test_idx[model_mask]

    model_start, model_end = true_parameters["row_ranges"][true_model]
    local_indices = model_test_indices - int(model_start)

    fractions = get_three_compartment_fractions(
        true_parameters,
        true_model,
    )[local_indices]

    f_ees = fractions[:, 0]
    f_vasc = fractions[:, 1]
    f_ic = fractions[:, 2]

    bin_edges = np.linspace(0, 1, TERNARY_N_BINS + 1)

    ees_bin = np.clip(
        np.digitize(f_ees, bin_edges) - 1,
        0,
        TERNARY_N_BINS - 1,
    )

    vasc_bin = np.clip(
        np.digitize(f_vasc, bin_edges) - 1,
        0,
        TERNARY_N_BINS - 1,
    )

    occupancy = np.zeros(
        (TERNARY_N_BINS, TERNARY_N_BINS),
        dtype=int,
    )

    np.add.at(occupancy, (ees_bin, vasc_bin), 1)

    prediction_counts = np.zeros(
        (
            TERNARY_N_BINS,
            TERNARY_N_BINS,
            len(CLASS_NAMES),
        ),
        dtype=int,
    )

    for seed in NOISE_SEEDS:
        predictions = variable_pred_by_seed[seed][model_mask]

        np.add.at(
            prediction_counts,
            (ees_bin, vasc_bin, predictions),
            1,
        )

    patches = []
    face_colours = []
    records = []

    for i in range(TERNARY_N_BINS):
        for j in range(TERNARY_N_BINS):
            n_unique = int(occupancy[i, j])

            if n_unique < PLOT_MIN_UNIQUE_SAMPLES:
                continue

            counts = prediction_counts[i, j]
            n_predictions = int(counts.sum())

            dominant_class = int(np.argmax(counts))
            dominant_proportion = counts[dominant_class] / n_predictions

            ees_low, ees_high = bin_edges[i:i + 2]
            vasc_low, vasc_high = bin_edges[j:j + 2]

            polygon = np.array([
                [ees_low, vasc_low],
                [ees_high, vasc_low],
                [ees_high, vasc_high],
                [ees_low, vasc_high],
            ])

            polygon = clip_polygon_to_simplex(polygon)

            if len(polygon) < 3:
                continue

            polygon_ees = polygon[:, 0]
            polygon_vasc = polygon[:, 1]
            polygon_ic = 1 - polygon_ees - polygon_vasc

            x, y = ternary_xy(
                polygon_ees,
                polygon_vasc,
                polygon_ic,
            )

            patches.append(
                Polygon(np.column_stack([x, y]), closed=True)
            )

            base_colour = CLASS_COLOURS[dominant_class]

            chance = 1 / len(CLASS_NAMES)
            strength = np.clip(
                (dominant_proportion - chance) / (1 - chance),
                0,
                1,
            )

            strength = 0.30 + 0.70 * strength
            displayed_colour = 1 - strength * (1 - base_colour)

            face_colours.append(displayed_colour)

            records.append({
                "true_model": true_model,
                "ees_bin_low": ees_low,
                "ees_bin_high": ees_high,
                "vasc_bin_low": vasc_low,
                "vasc_bin_high": vasc_high,
                "n_unique_test_signals": n_unique,
                "dominant_prediction": CLASS_NAMES[dominant_class],
                "dominant_proportion": dominant_proportion,
            })

    collection = PatchCollection(
        patches,
        facecolors=face_colours,
        edgecolors=(1, 1, 1, 0.2),
        linewidths=0.1,
    )

    ax.add_collection(collection)

    draw_ternary_frame(ax)
    ax.set_title(SHORT_NAME_MAP[true_model])

    return records

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8),
)

axes = axes.ravel()
fraction_space_rows = []

for ax, model_name in zip(
    axes[:5],
    THREE_COMPARTMENT_MODELS,
):
    rows = plot_fraction_space(
        ax,
        model_name,
    )

    fraction_space_rows.extend(rows)


# Use the sixth panel for the legend.
axes[5].axis("off")

legend_handles = [
    Patch(
        facecolor=CLASS_COLOURS[i],
        edgecolor="none",
        label=class_name,
    )
    for i, class_name in enumerate(CLASS_NAMES)
]

axes[5].legend(
    handles=legend_handles,
    title="Dominant predicted model",
    loc="center",
    frameon=False,
)

fig.suptitle(
    "Model selection across three-compartment fraction space",
)

fig.tight_layout()

fraction_space_df = pd.DataFrame(fraction_space_rows)

fraction_space_df.to_csv(
    TABLE_DIR / "mlp_fraction_space_variable_snr30_70.csv",
    index=False,
)

fig.savefig(
    FIGURE_DIR / "mlp_fraction_space_variable_snr30_70.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR / "mlp_fraction_space_variable_snr30_70.pdf",
    bbox_inches="tight",
)

plt.show()

### End of notebook 03
